# Student Performance Prediction

**Python • scikit-learn • Pandas • Matplotlib • XGBoost**

---

### What this project does
*Student Performance Prediction* is a beginner-friendly machine learning project that predicts a student's exam score based on their study habits and personal background information.

### 🎯 Problem It Solves
In education, identifying struggling students early can make a huge difference. This project builds a smart prediction system that estimates how well a student will perform in their exam — giving educators a data-driven tool to intervene early and support at-risk students.

---
## Step 2: Install & Import Libraries

Run this once if packages are missing.

In [ ]:
# Uncomment only if you need to install packages
# !pip install -r requirements.txt

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


warnings.filterwarnings("ignore")
plt.style.use("seaborn-v0_8-whitegrid")

print("All libraries loaded successfully!")

---
## Step 2: Load the Data

We read the CSV file from the `data/` folder and show the first few rows.

In [ ]:
DATA_PATH = os.path.join("data", "StudentPerformanceFactors.csv")
df = pd.read_csv(DATA_PATH)
print(f"Data loaded: {df.shape[0]} rows, {df.shape[1]} columns")
df.head()

---
## Step 3: Explore the Data (EDA)

Before building models, we check:
- Dataset size
- Missing values (then **clean** them before modeling)
- Basic statistics
- Simple charts

In [ ]:
print("Shape (rows, columns):", df.shape)

In [ ]:
df.info()

In [ ]:
print("\nColumn names:")
print(df.columns.tolist())

In [ ]:
# Check missing values
print(df.isnull().sum())

In [ ]:
# Fill missing values (use most common category)

for col in ["Teacher_Quality", "Parental_Education_Level", "Distance_from_Home"]:
    df[col] = df[col].fillna(df[col].mode()[0])
    
print(df.isnull().sum())

In [ ]:
print("\nBasic statistics:")
df.describe()

In [ ]:
# Chart 1: Distribution of exam scores
plt.figure(figsize=(7, 4))
plt.hist(df["Exam_Score"], bins=25, color="steelblue", edgecolor="white")
plt.title("Distribution of Exam Scores")
plt.xlabel("Exam Score")
plt.ylabel("Number of Students")
plt.show()



In [ ]:
# Chart 2: Study hours vs exam score
plt.figure(figsize=(7, 4))
plt.scatter(df["Hours_Studied"], df["Exam_Score"], alpha=0.5, color="darkorange")
plt.title("Study Hours vs Exam Score")
plt.xlabel("Hours Studied")
plt.ylabel("Exam Score")
plt.show()

---
## Step 4: Feature Engineering

**Feature engineering** means creating new useful columns from existing data.

| New Feature | Formula / Logic | Why it helps |
|-------------|-------------------|--------------|
| `StudyEfficiency` | Previous_Scores ÷ (Hours_Studied + 1) | Students who score high with fewer hours |
| `GoodSleep` | 1 if sleep is 6–8 hours, else 0 | Healthy sleep may help performance |
| `StudySleepBalance` | Hours_Studied × Sleep_Hours | Balance between study and rest |

We also convert text columns (like Gender) into numbers using **one-hot encoding**.

In [ ]:
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor


In [ ]:
data = df.copy()

# --- Create new features ---
data["StudyEfficiency"] = data["Previous_Scores"] / (data["Hours_Studied"] + 1)
data["GoodSleep"] = ((data["Sleep_Hours"] >= 6) & (data["Sleep_Hours"] <= 8)).astype(int)
data["StudySleepBalance"] = data["Hours_Studied"] * data["Sleep_Hours"]

# --- Convert text columns to numbers (one-hot encoding) ---
categorical_cols = [
    "Gender",
    "Parental_Involvement",
    "Access_to_Resources",
    "Extracurricular_Activities",
    "Motivation_Level",
    "Internet_Access",
    "Family_Income",
    "Teacher_Quality",
    "School_Type",
    "Peer_Influence",
    "Parental_Education_Level",
    "Distance_from_Home",
    "Learning_Disabilities",
]

data = pd.get_dummies(data, columns=categorical_cols, drop_first=True)



In [ ]:
# --- Define X (features) and y (target) ---
TARGET = "Exam_Score"
X = data.drop(columns=[TARGET])
y = data[TARGET]

print(f"Number of features: {X.shape[1]}")
print(f"Target variable: {TARGET}")
X.head()

---
## Step 5: Train / Test Split + Feature Scaling

- **80% training** → model learns patterns from this data
- **20% testing** → model is evaluated on unseen data
- **Scaling** → puts numeric features on a similar scale (helps Linear Regression)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)



In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)



In [ ]:
print(f"Training samples: {len(X_train)}")
print(f"Testing samples:  {len(X_test)}")

---
## Step 6: Train 3 Models & Compare

We train and compare:
1. **Linear Regression** — simple, fast baseline
2. **Random Forest** — ensemble of decision trees
3. **XGBoost** — powerful gradient boosting model

### Metrics explained (easy version)
| Metric | Meaning | Better when |
|--------|---------|-------------|
| **MAE** | Average prediction error (in score points) | Lower |
| **RMSE** | Like MAE, but punishes large errors more | Lower |
| **R²** | % of score variation the model explains (max = 1.0) | Higher |

In [ ]:
def evaluate_model(name, y_true, y_pred):
    """Calculate MAE, RMSE, and R² for a model."""
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)

    print(f"\n{name}")
    print(f"  MAE  = {mae:.2f}  (average error in points)")
    print(f"  RMSE = {rmse:.2f}  (penalizes big errors)")
    print(f"  R²   = {r2:.2f}   (1.0 = perfect)")

    return {"Model": name, "MAE": mae, "RMSE": rmse, "R²": r2}


results = []



In [ ]:
# --- Model 1: Linear Regression ---
lr_model = LinearRegression()
lr_model.fit(X_train_scaled, y_train)
lr_pred = lr_model.predict(X_test_scaled)
results.append(evaluate_model("Linear Regression", y_test, lr_pred))


In [ ]:
# --- Model 2: Random Forest ---
rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)
rf_pred = rf_model.predict(X_test)
results.append(evaluate_model("Random Forest", y_test, rf_pred))

In [ ]:
# --- Model 3: XGBoost ---
xgb_model = XGBRegressor(
    n_estimators=100, learning_rate=0.1, max_depth=5,
    random_state=42, verbosity=0
)
xgb_model.fit(X_train, y_train)
xgb_pred = xgb_model.predict(X_test)
results.append(evaluate_model("XGBoost", y_test, xgb_pred))

In [ ]:
# --- Comparison table ---
results_df = pd.DataFrame(results).sort_values("R²", ascending=False)
print("\n" + "=" * 40)
print("MODEL COMPARISON TABLE")
print("=" * 40)
results_df

In [ ]:
# Visual comparison of R² scores
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

colors = ["#4C72B0", "#DD8452", "#55A868"]

axes[0].bar(results_df["Model"], results_df["R²"], color=colors)
axes[0].set_title("R² Score (Higher is Better)")
axes[0].set_ylabel("R²")
axes[0].set_ylim(0, 1)
axes[0].tick_params(axis="x", rotation=15)

axes[1].bar(results_df["Model"], results_df["MAE"], color=colors)
axes[1].set_title("MAE (Lower is Better)")
axes[1].set_ylabel("MAE")
axes[1].tick_params(axis="x", rotation=15)

plt.tight_layout()
plt.show()

---
## Step 7: Cross-Validation (Prevent Overfitting)

**Why?** A model can memorize training data and fail on new students.

**5-Fold Cross-Validation** splits training data into 5 parts.  
The model trains 5 times — each time using a different part as validation.

If R² scores are **similar across all 5 folds**, the model generalizes well.

In [ ]:
models_cv = {
    "Linear Regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(n_estimators=100, random_state=42),
    "XGBoost": XGBRegressor(
        n_estimators=100, learning_rate=0.1, max_depth=5,
        random_state=42, verbosity=0
    ),
}

print("5-Fold Cross-Validation Results (R² scores):\n")
cv_summary = []



In [ ]:
for name, model in models_cv.items():
    X_cv = X_train_scaled if name == "Linear Regression" else X_train
    scores = cross_val_score(model, X_cv, y_train, cv=5, scoring="r2")

    print(f"{name}")
    print(f"  Fold scores: {np.round(scores, 3)}")
    print(f"  Mean R²:     {scores.mean():.3f}  (+/- {scores.std():.3f})\n")

    cv_summary.append({
        "Model": name,
        "Mean R²": scores.mean(),
        "Std R²": scores.std(),
    })



In [ ]:
pd.DataFrame(cv_summary).sort_values("Mean R²", ascending=False)

---
## Step 8: Hyperparameter Tuning (Improve XGBoost)

We use **GridSearchCV** to try different XGBoost settings and pick the best combination.

In [ ]:
param_grid = {
    "n_estimators": [100, 200, 300],
    "max_depth": [3, 5, 7],
    "learning_rate": [0.05, 0.1, 0.15],
    "subsample": [0.8, 1.0],
}

xgb_base = XGBRegressor(random_state=42, verbosity=0)

grid_search = GridSearchCV(
    estimator=xgb_base,
    param_grid=param_grid,
    cv=5,
    scoring="r2",
    n_jobs=-1,
)

print("Tuning XGBoost... (this may take 1–2 minutes)")
grid_search.fit(X_train, y_train)

print("\nBest parameters found:")
for key, value in grid_search.best_params_.items():
    print(f"  {key}: {value}")
print(f"\nBest cross-validation R²: {grid_search.best_score_:.3f}")

---
## Step 9: Final Results

Evaluate the **tuned XGBoost** model on the test set and visualize predictions.

In [ ]:
best_model = grid_search.best_estimator_
final_pred = best_model.predict(X_test)

final_mae = mean_absolute_error(y_test, final_pred)
final_rmse = np.sqrt(mean_squared_error(y_test, final_pred))
final_r2 = r2_score(y_test, final_pred)

print("=" * 45)
print("  FINAL TUNED XGBOOST — TEST SET RESULTS")
print("=" * 45)
print(f"  MAE  = {final_mae:.2f}")
print(f"  RMSE = {final_rmse:.2f}")
print(f"  R²   = {final_r2:.2f}")
print("=" * 45)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Actual vs Predicted scatter plot
axes[0].scatter(y_test, final_pred, alpha=0.6, color="steelblue")
axes[0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], "r--", lw=2)
axes[0].set_xlabel("Actual Exam Score")
axes[0].set_ylabel("Predicted Exam Score")
axes[0].set_title(f"Actual vs Predicted (R² = {final_r2:.2f})")

# Residual plot (errors)
residuals = y_test - final_pred
axes[1].scatter(final_pred, residuals, alpha=0.6, color="coral")
axes[1].axhline(0, color="red", linestyle="--")
axes[1].set_xlabel("Predicted Exam Score")
axes[1].set_ylabel("Residual (Actual - Predicted)")
axes[1].set_title("Residual Plot")

plt.tight_layout()
plt.show()

In [ ]:
# Top 10 most important features
importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": best_model.feature_importances_,
}).sort_values("Importance", ascending=False).head(10)

plt.figure(figsize=(8, 5))
plt.barh(importance["Feature"], importance["Importance"], color="seagreen")
plt.gca().invert_yaxis()
plt.title("Top 10 Important Features (Tuned XGBoost)")
plt.xlabel("Importance")
plt.show()

importance

---
## Step 10: Project Summary

### What we built
| Component | Details |
|-----------|--------|
| **Data source** | Kaggle — Student Performance Factors |
| **Target** | `Exam_Score` |
| **Feature engineering** | StudyEfficiency, GoodSleep, StudySleepBalance + one-hot encoding |
| **Models compared** | Linear Regression, Random Forest, XGBoost |
| **Evaluation metrics** | MAE, RMSE, R² |
| **Validation** | 5-fold cross-validation |
| **Tuning** | GridSearchCV on XGBoost hyperparameters |

###  Final Result
XGBoost, after hyperparameter tuning, delivers the best performance with an R² score of approximately 0.74, meaning the model explains 74% of the variation in student exam scores — a strong and realistic result for this type of dataset.